# 00_ListStage2Products: what does stage 2 (`stage2-recalibrate`) produce?

Starting point of the 06_AnaStep2 series. No assumption is made on dataset names: we list what the stage-2 output run actually contains, then open one example of each FGCM-related product.

Facts taken from the w_2026_43 pipeline yaml (`doc/rubin-pipeline-info/pipeline_drp_pipe_LSSTCam_DRP.yaml`):
- `fgcmBuildFromIsolatedStar` reads `isolated_star_association`, `isolated_star` and `preliminary_visit_summary`;
- `fgcmFitCycle` is run with `doMultipleCycles: true`; `fgcmOutputProducts` follows;
- `updateVisitSummary` takes `fgcmPhotoCalibCatalog` as `photo_calib_overrides_global`, the refitted PSF models (`refit_psf_star`, `refit_psf_models`) and the gbdes astrometric solution, and writes `visit_summary`;
- stage 2 also refits the PSF (`refitPsfModelDetector`), which matters for the visits whose stage-1 PSF model failed (notebook 03).

Run with the `lsst` kernel at CC-IN2P3.

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-04
- last update : 2026-11-04
- `quantum graph for step2a` : https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_43/drp_pipe/LSSTCam/DRP/pipeline_drp_pipe_LSSTCam_DRP_step2a-recalibrate-global.pdf

In [ ]:
import os
import sys
from collections import Counter
from pathlib import Path

from IPython.display import display, Image, Markdown

import numpy as np
import pandas as pd

SRC = Path('../../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from rubincc import monster as mon

NB_TAG = '00_ListStage2Products'
DATA_DIR = Path(f'data_{NB_TAG}')
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
REPO = 'main'
USER = os.getenv('USER')

# stage-2 collection of the run to inspect (switch to the 20261103 run when it is available)
STAGE2 = f'u/{USER}/LSSTCam/runcosmos01_20261026_tract9813patch50_RunCosmosDemo/stage2'

In [ ]:
def ShowOneImage(filename_img, title = None, caption = None):
    """
    """
    print(os.path.abspath(filename_img))
    print(os.path.exists(filename_img))
    with open(filename_img, "rb") as f:
        data = f.read()
    print("File size:", len(data), "bytes")
    print("PNG signature:", data[:8])
    if title is not None:
        display(Markdown(f"### {title}"))
    display(Image(data=data))
    if caption is not None:
        display(Markdown(f"### {caption}"))

In [ ]:
# figure with quantum diagram
filename_diagram = "../fig_diagrams/pipeline_drp_pipe_LSSTCam_DRP_step2a-recalibrate-global.png"
ShowOneImage(filename_diagram,title="DRP Step2a")

## 1. Collections of the stage-2 output

`stage2` is a chained collection: its first children are the output RUN collections, followed by the inputs (stage-1 outputs, calibrations, ...). We only look at the RUN collections written by stage 2.

In [ ]:
butler = mon.get_butler(REPO, collections=STAGE2)
reg = butler.registry

chain = list(reg.getCollectionChain(STAGE2))
print('children of the chain:')
for name in chain:
    print('  ', reg.getCollectionType(name).name, name)

# RUN collections written by stage 2 (the name contains 'stage2')
stage2_runs = [n for n in chain if reg.getCollectionType(n).name == 'RUN' and 'stage2' in n]
print('stage-2 RUN collections:', stage2_runs)

## 2. Dataset types produced by stage 2

In [ ]:
# one query over all dataset types of the stage-2 RUN collections
try:
    refs = butler.query_all_datasets(stage2_runs, limit=200000)
except Exception as exc:
    print('query_all_datasets failed:', exc)
    refs = []
    for dt in reg.queryDatasetTypes('*'):
        refs += list(reg.queryDatasets(dt, collections=stage2_runs, findFirst=False))

counts = Counter(ref.datasetType.name for ref in refs)
first = {}
for ref in refs:
    first.setdefault(ref.datasetType.name, ref)

table = pd.DataFrame([{
    'dataset_type': name,
    'n_datasets': n,
    'dimensions': ', '.join(first[name].datasetType.dimensions.names),
    'storage_class': first[name].datasetType.storageClass_name,
} for name, n in counts.items()]).sort_values('dataset_type').reset_index(drop=True)
table.to_csv(DATA_DIR / f'{NB_TAG}_dataset_types.csv', index=False)
print(len(table), 'dataset types,', len(refs), 'datasets')
table.to_string(index=False)

In [ ]:
# the products of the FGCM chain and of the final recalibration
WANTED = ('fgcm', 'photocalib', 'photo_calib', 'visit_summary', 'refit_psf', 'gbdes', 'isolated_star', 'recalibrated')
focus = table[table['dataset_type'].str.lower().apply(lambda s: any(w in s for w in WANTED))]
focus.to_string(index=False)

## 3. One example of each FGCM-related product

Only a description is printed (type, size, columns), to learn the structure before analysing it.

In [ ]:
def describe(obj):
    # short description of whatever the Butler returned
    info = {'python_type': type(obj).__name__}
    if isinstance(obj, pd.DataFrame):
        info['shape'] = obj.shape
        info['columns'] = list(obj.columns)[:40]
    elif hasattr(obj, 'colnames'):
        info['shape'] = (len(obj), len(obj.colnames))
        info['columns'] = list(obj.colnames)[:40]
    elif hasattr(obj, 'asAstropy'):
        tab = obj.asAstropy()
        info['shape'] = (len(tab), len(tab.colnames))
        info['columns'] = list(tab.colnames)[:40]
    elif hasattr(obj, 'shape'):
        info['shape'] = obj.shape
    else:
        info['public_attributes'] = [a for a in dir(obj) if not a.startswith('_')][:30]
    return info


fgcm_types = [n for n in table['dataset_type'] if 'fgcm' in n.lower()]
print('FGCM dataset types:', fgcm_types)
for name in fgcm_types:
    ref = first[name]
    print('---', name, '|', ref.dataId)
    try:
        print(describe(butler.get(ref)))
    except Exception as exc:
        print('   could not read:', repr(exc))